In [1]:
import time
import math
import random

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

I0000 00:00:1795516608.535623   21494 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795516608.540450   21494 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795516609.151745   21494 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795516616.361507   21494 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [ ]:
MODEL_PATH = "face_landmarker.task"
CAMERA_ID = 0

IRIS_L = 468
IRIS_R = 473
NOSE_TIP = 1

BEAM_OPENING_DEG = 20
BEAM_LENGTH = 10

YAW_GAIN = 2.5
PITCH_GAIN = 2.5
ANGLE_LIMIT = 0.5
SMOOTH = 0.3

BEAM_WIDTH = 0.12
FLARE_RADIUS = 0.22
FLICKER = 0.15
GLOW_SIGMA_REL = 0.2
GLOW_STRENGTH = 1.0

RISE_SPEED = 0.2
FALL_SPEED = 0.1

BEAM_LAYERS = [
    (1.00, (0, 90, 255)),
    (0.60, (0, 200, 255)),
    (0.25, (255, 255, 255)),
]
FLARE_LAYERS = [
    (1.00, (0, 120, 255)),
    (0.60, (0, 220, 255)),
    (0.30, (255, 255, 255)),
]


In [3]:
def create_detector():
    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options =  vision.FaceLandmarkerOptions(
        base_options = base_options,
        running_mode = vision.RunningMode.VIDEO,
        num_faces = 1
    )
    return vision.FaceLandmarker.create_from_options(options)

In [4]:
def get_face_points(result, w, h):
    if not result.face_landmarks:
        return None

    points = result.face_landmarks[0]

    left = points[IRIS_L]
    right = points[IRIS_R]
    nose = points[NOSE_TIP]
    
    if right.x < left.x:
        left, right = right, left 

    left_px = (left.x * w, left.y * h)
    right_px = (right.x * w, right.y * h)
    nose_px = (nose.x * w, nose.y * h)

    return left_px, right_px, nose_px


In [5]:
def to_int(p):
    return(int(round(p[0])), int(round(p[1])))

In [6]:
def beam_end(eye, angle, length):
    return (
        eye[0] + length * math.cos(angle),
        eye[1] + length * math.sin(angle)
    )

In [7]:
def clamp(x, lo, hi):
    return max(lo, min(hi, x))

In [8]:
def face_to_img_angle(dx, dy, theta):
    u = (math.cos(theta), math.sin(theta))
    v = (-math.sin(theta), math.cos(theta))
    ix = dx * u[0] + dy * v[0]
    iy = dx * u[1] + dy * v[1]
    return math.atan2(iy, ix)

In [9]:
def draw_beams(frame, beams, eye_mod, intensity):
    h, w = frame.shape[:2]
    layer = np.zeros_like(frame)

    flicker = random.uniform(1 - FLICKER, 1 + FLICKER)
    width = eye_mod * BEAM_WIDTH * flicker

    for eye, end in beams:
        tip = (
            eye[0] + (end[0] - eye[0]) * intensity,
            eye[1] + (end[1] - eye[1]) * intensity
        )
        p0, p1 = to_int(eye), to_int(tip)

        for scale, color in BEAM_LAYERS:
            t = max(1, int(round(width * scale)))
            cv2.line(layer, p0, p1, color, t, cv2.LINE_AA)

        for scale, color in FLARE_LAYERS:
            rad = max(1, int(round(eye_mod * FLARE_RADIUS * flicker * scale)))
            cv2.circle(layer, p0, rad, color, -1, cv2.LINE_AA)

    layer = cv2.convertScaleAbs(layer, alpha=intensity)

    small = cv2.resize(layer, (w // 2, h // 2), interpolation=cv2.INTER_AREA)
    sigma = max(2.0, eye_mod * GLOW_SIGMA_REL / 2)
    small = cv2.GaussianBlur(small, (0, 0), sigma)
    glow = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    glow = cv2.convertScaleAbs(glow, alpha=GLOW_STRENGTH)

    out = cv2.add(frame, glow)
    out = cv2.add(out, layer)
    return out




In [10]:
detector = create_detector()

v = cv2.VideoCapture(CAMERA_ID)

t_0 = time.perf_counter()

smooth_yaw = None
smooth_pitch = None
yaw_0 = 0.0
pitch_0 = 0.0

calibrated = False

intensity = 0.0
last_beams = None

while True:
    ret, frame = v.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    timestamp = int((time.perf_counter() - t_0) * 1000)
    r = detector.detect_for_video(mp_img, timestamp)

    face = get_face_points(r, w, h)

    face_ok = face is not None
    if face_ok:
        eye_l, eye_r, nose = face
        
        eye_v = (eye_r[0] - eye_l[0], eye_r[1] - eye_l[1])
        eye_mod = math.hypot(eye_v[0], eye_v[1])
        theta = math.atan2(eye_v[1], eye_v[0])

        u = (math.cos(theta), math.sin(theta))
        v_axis = (-math.sin(theta), math.cos(theta))

        mid = ((eye_l[0] + eye_r[0])/2, (eye_l[1]+eye_r[1])/2)
        nose_v = (nose[0] - mid[0], nose[1] - mid[1])
        yaw = (nose_v[0] * u[0] + nose_v[1] * u[1]) / eye_mod
        pitch = (nose_v[0] * v_axis[0] + nose_v[1] * v_axis[1]) / eye_mod

        if smooth_yaw is None:
            smooth_yaw, smooth_pitch = yaw, pitch
        else:
            smooth_yaw += SMOOTH * (yaw - smooth_yaw)
            smooth_pitch += SMOOTH * (pitch - smooth_pitch)

        yaw = clamp(smooth_yaw - yaw_0, -ANGLE_LIMIT, ANGLE_LIMIT)
        pitch = clamp(smooth_pitch - pitch_0, -ANGLE_LIMIT, ANGLE_LIMIT)

        phi = math.radians(BEAM_OPENING_DEG)
        shift_x = YAW_GAIN * yaw
        shift_y = PITCH_GAIN * pitch

        dir_l = (-math.sin(phi) + shift_x, math.cos(phi) + shift_y)
        dir_r = (math.sin(phi) + shift_x, math.cos(phi) + shift_y)
 
        angle_l = face_to_img_angle(dir_l[0], dir_l[1], theta)
        angle_r = face_to_img_angle(dir_r[0], dir_r[1], theta)

        length = BEAM_LENGTH * eye_mod

        end_l = beam_end(eye_l, angle_l, length)
        end_r = beam_end(eye_r, angle_r, length)

        last_beams = ([(eye_l, end_l), (eye_r, end_r)], eye_mod)

    else:
        smooth_yaw = None
        smooth_pitch = None

    if face_ok:
        intensity = min(1.0, intensity + RISE_SPEED)
    else:
        intensity = max(0, intensity - FALL_SPEED)

    if intensity > 0 and last_beams is not None:
        frame = draw_beams(frame, last_beams[0], last_beams[1], intensity)

    if face_ok and not calibrated:
        cv2.putText(
            frame, "Mira de frente y pulsa C para calibrar", (20, h - 20),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 255), 2
        )
    elif not face_ok:
        cv2.putText(
            frame, "Sin cara", (20, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2
        )


    cv2.imshow("Filtro", frame)

    key = cv2.waitKey(1) & 0xFF
    if key in (ord("q"), 27):
            break
    elif key == ord("c") and face_ok:
        yaw_0 = smooth_yaw
        pitch_0 = smooth_pitch
        calibrated = True
    
v.release()
cv2.destroyAllWindows()
detector.close()

W0000 00:00:1795516622.238389   21602 face_landmarker_graph.cc:181] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1795516622.285840   21606 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795516622.336103   21606 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
